# Vector Search in Practice

This notebook demonstrates how to implement vector search and build a RAG (Retrieval-Augmented Generation) pipeline using a reviews dataset.

## 1. Data Loading and Preparation

In [ ]:
## Download the dataset from the source - https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews/ in a zip and extract in directory archive/


import numpy as np # linear algebra
import pandas as pd
df = pd.read_csv("/content/Reviews.csv")

df.dataframeName = 'Reviews.csv'

nRow, nCol = df.shape

print(f'There are {nRow} rows and {nCol} columns')
df.head(2)

### Data clean up

In [ ]:
df = df.drop(columns=['Id', 'ProfileName', 'Time', 'UserId'], axis=1)

df = df.drop(columns=['HelpfulnessNumerator', 'HelpfulnessDenominator'], axis=1)
df.dropna(subset=['Text','Summary'],inplace=True)
print("After dropping", df.shape[0])

# remove all duplicate rows based on (ingredients, directions) together
df = df.drop_duplicates(subset=['Text', 'Summary'])

df.nunique()

# Display the first few rows
df.head()

## 2. Data Exploration (EDA)

# Install required visualization libraries if not already installed

In [ ]:
!pip install matplotlib seaborn wordcloud

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Set the style for plots
sns.set(style="whitegrid")
plt.figure(figsize=(12, 6))

In [ ]:
# Check for missing values
print("Missing values in each column:")
print(df.isnull().sum())

# Basic statistics of the dataset
print("\nBasic statistics:")
print(df.describe(include='all'))

In [ ]:
# Distribution of ratings
plt.figure(figsize=(10, 6))
sns.countplot(x='Score', data=df)
plt.title('Distribution of Review Scores')
plt.xlabel('Score')
plt.ylabel('Count')
plt.show()

In [ ]:
# Calculate text length

df['text_length'] = df['Text'].apply(len)
df['summary_length'] = df['Summary'].apply(len)

# Plot text length distribution
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.histplot(df['text_length'], kde=True)
plt.title('Distribution of Review Text Length')
plt.xlabel('Length')
plt.xlim(0, df['text_length'].quantile(0.95))  # Limit x-axis to 95th percentile for better visualization

plt.subplot(1, 2, 2)
sns.histplot(df['summary_length'], kde=True)
plt.title('Distribution of Summary Length')
plt.xlabel('Length')
plt.xlim(0, df['summary_length'].quantile(0.95))  # Limit x-axis to 95th percentile

plt.tight_layout()
plt.show()

In [ ]:
# Word cloud for review text
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Combine all review texts
all_text = ' '.join(df['Text'].sample(n=10000).tolist())

# Generate word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white', max_words=100).generate(all_text)

# Display the word cloud
plt.figure(figsize=(12, 8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Review Texts')
plt.show()

## 3. Embedding Methods with Sentence-Transformers

# Install required libraries





We sample 50,000 reviews to make the embedding process faster and more memory-friendly.

In [ ]:
!pip install sentence-transformers faiss-cpu langchain

In [ ]:
sampled_df = df.sample(n=50000,random_state=40).reset_index(drop=True)
sampled_df.head(2)

In [ ]:
from sentence_transformers import SentenceTransformer
import pickle
import os

# Initialize the model
model = SentenceTransformer('all-MiniLM-L6-v2')



embeddings_file = 'review_embeddings.pkl'

# Check if embeddings file already exists
if os.path.exists(embeddings_file):
    print(f"Loading embeddings from {embeddings_file}...")
    with open(embeddings_file, 'rb') as f:
        embeddings = pickle.load(f)
    print(f"Embeddings loaded successfully! Shape: {embeddings.shape}")
else:
    # Generate embeddings for the review texts
    print("Generating embeddings for reviews...")
    embeddings = model.encode(sampled_df['Text'].tolist(), show_progress_bar=True, batch_size=1024)
    print(f"Embedding shape: {embeddings.shape}")

    # Save embeddings to pickle file
    print(f"Saving embeddings to {embeddings_file}...")
    with open(embeddings_file, 'wb') as f:
        pickle.dump(embeddings, f)
    print(f"Embeddings saved successfully!")


In [ ]:
# Visualize the embeddings using PCA
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Reduce dimensions to 2D for visualization
pca = PCA(n_components=2)
reduced_embeddings = pca.fit_transform(embeddings)

# Plot the reduced embeddings
plt.figure(figsize=(10, 8))
plt.scatter(reduced_embeddings[:, 0], reduced_embeddings[:, 1], alpha=0.5)
plt.title('PCA Visualization of Review Embeddings')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.show()

In [ ]:
# Color the PCA plot by review score
plt.figure(figsize=(10, 8))
scatter = plt.scatter(reduced_embeddings[:, 0], reduced_embeddings[:, 1],
                     c=sampled_df['Score'], cmap='viridis', alpha=0.6)
plt.colorbar(scatter, label='Review Score')
plt.title('PCA Visualization of Review Embeddings by Score')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.show()

## 4. Build the RAG Pipeline

### Create FAISS Vector Index

This cell installs the `faiss-cpu` library, which is essential for efficient similarity search and vector index creation.







In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
import numpy as np

# Normalize the vectors to unit length for cosine similarity
embeddings_normalized = embeddings.copy()
faiss.normalize_L2(embeddings_normalized)

# Create the FAISS index
dimension = embeddings.shape[1]  # Dimension of the embeddings
index = faiss.IndexFlatIP(dimension)  # Inner product for cosine similarity with normalized vectors

# Add the vectors to the index
index.add(embeddings_normalized)

print(f"FAISS index contains {index.ntotal} vectors")

### Set Up FAISS Retrieval

We creates a helper function that turns a query into an embedding and retrieves the most similar reviews from the FAISS index.

In [ ]:
def search_reviews(query, top_k=5):
    """
    Search for reviews similar to the query.

    Args:
        query (str): The search query
        top_k (int): Number of results to return

    Returns:
        list: Top k similar reviews with their scores
    """
    # Encode the query
    query_vector = model.encode([query])

    # Normalize the query vector
    faiss.normalize_L2(query_vector)

    # Search the index
    scores, indices = index.search(query_vector, top_k)

    # Get the results
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            'score': scores[0][i],
            'text': sampled_df.iloc[idx]['Text'],
            'summary': sampled_df.iloc[idx]['Summary'],
            'product_score': sampled_df.iloc[idx]['Score']
        })

    return results

### Example Queries

In [ ]:
# Example queries
example_queries = [
    "delicious chocolate",
    "terrible taste",
    "great customer service",
    "expensive but worth it",
    "would not recommend"
]

# Search for each query
for query in example_queries:
    print(f"\nQuery: '{query}'")
    results = search_reviews(query, top_k=3)

    for i, result in enumerate(results):
        print(f"\nResult {i+1} (Score: {result['score']:.4f}, Product Rating: {result['product_score']})")
        print(f"Summary: {result['summary']}")
        print(f"Text: {result['text'][:200]}...")

### Integrate with LLM using LangChain

We use HuggingFace embeddings + LangChain’s FAISS integration to build a search-ready vector store.

In [ ]:
# Install required packages
!pip install langchain langchain-community

In [ ]:
from langchain.vectorstores import FAISS as LangchainFAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document

# Create documents from the sampled dataframe
documents = []
for i, row in sampled_df.iterrows():
    doc = Document(
        page_content=row['Text'],
        metadata={
            'summary': row['Summary'],
            'score': row['Score']
        }
    )
    documents.append(doc)

In [ ]:
# Initialize the HuggingFace embeddings
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create the vector store
vector_store = LangchainFAISS.from_documents(documents, embeddings_model)

# Test the vector store
query = "delicious chocolate"
docs = vector_store.similarity_search(query, k=3)

print(f"Query: '{query}'")
for i, doc in enumerate(docs):
    print(f"\nResult {i+1}")
    print(f"Summary: {doc.metadata['summary']}")
    print(f"Product Rating: {doc.metadata['score']}")
    print(f"Text: {doc.page_content[:200]}...")

In [ ]:
# Install an open-source LLM package
!pip install ctransformers

In [ ]:
from langchain.llms import CTransformers
from langchain.chains import RetrievalQA

# Initialize the LLM
# Note: You'll need to download the model first or use a different model
# This is just an example - you might need to adjust based on available models
try:
    llm = CTransformers(
        model="TheBloke/Llama-2-7B-Chat-GGML",
        model_type="llama",
        max_new_tokens=512,
        temperature=0.7
    )

    # Create the retrieval QA chain
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vector_store.as_retriever(search_kwargs={"k": 3}),
        return_source_documents=True
    )

    # Example query
    query = "What do people say about chocolate products?"
    result = qa_chain({"query": query})

    print(f"Query: {query}")
    print(f"\nAnswer: {result['result']}")
    print("\nSource Documents:")
    for i, doc in enumerate(result["source_documents"]):
        print(f"\nDocument {i+1}:")
        print(f"Summary: {doc.metadata['summary']}")
        print(f"Text: {doc.page_content[:200]}...")
except Exception as e:
    print(f"Error initializing LLM: {e}")
    print("\nAlternative: You can use OpenAI or other API-based LLMs if available.")

## 5. Evaluation

### Create Query-Relevance Pairs for Evaluation

In [ ]:
# Create some example query-relevance pairs for evaluation
# In a real scenario, these would be manually created or derived from user interactions
evaluation_queries = [
    {
        "query": "chocolate taste",
        "relevant_indices": []  # We'll fill these based on keyword matching for demonstration
    },
    {
        "query": "poor quality product",
        "relevant_indices": []
    },
    {
        "query": "excellent customer service",
        "relevant_indices": []
    }
]

# For demonstration, let's find relevant reviews using simple keyword matching
import re

for query_item in evaluation_queries:
    query_keywords = query_item["query"].lower().split()

    for i, row in sampled_df.iterrows():
        text = row["Text"].lower()
        summary = row["Summary"].lower()

        # Check if all keywords are in the text or summary
        if all(keyword in text or keyword in summary for keyword in query_keywords):
            query_item["relevant_indices"].append(i)

    # Limit to top 10 relevant indices for simplicity
    query_item["relevant_indices"] = query_item["relevant_indices"][:10]

    print(f"Query: '{query_item['query']}'")
    print(f"Found {len(query_item['relevant_indices'])} relevant reviews")

### Implement Precision@K and Recall@K

We implement standard ranking metrics to measure how well the vector search retrieves relevant items


In [ ]:
def precision_at_k(retrieved_indices, relevant_indices, k):
    """
    Calculate Precision@K

    Args:
        retrieved_indices: List of retrieved document indices
        relevant_indices: List of relevant document indices
        k: Number of top results to consider

    Returns:
        float: Precision@K score
    """
    if not retrieved_indices or k == 0:
        return 0.0

    # Consider only top k results
    retrieved_indices = retrieved_indices[:k]

    # Count relevant documents in the retrieved set
    relevant_retrieved = len(set(retrieved_indices) & set(relevant_indices))

    return relevant_retrieved / min(k, len(retrieved_indices))

def recall_at_k(retrieved_indices, relevant_indices, k):
    """
    Calculate Recall@K

    Args:
        retrieved_indices: List of retrieved document indices
        relevant_indices: List of relevant document indices
        k: Number of top results to consider

    Returns:
        float: Recall@K score
    """
    if not relevant_indices or not retrieved_indices:
        return 0.0

    # Consider only top k results
    retrieved_indices = retrieved_indices[:k]

    # Count relevant documents in the retrieved set
    relevant_retrieved = len(set(retrieved_indices) & set(relevant_indices))

    return relevant_retrieved / len(relevant_indices)

In [ ]:
# Evaluate the vector search
k_values = [1, 3, 5, 10]
results = []

for query_item in evaluation_queries:
    query = query_item["query"]
    relevant_indices = query_item["relevant_indices"]

    print(f"\nEvaluating query: '{query}'")

    # Encode the query
    query_vector = model.encode([query])
    faiss.normalize_L2(query_vector)

    # Get the maximum k value
    max_k = max(k_values)

    # Search the index
    scores, indices = index.search(query_vector, max_k)
    retrieved_indices = indices[0].tolist()

    # Calculate metrics for different k values
    for k in k_values:
        p_at_k = precision_at_k(retrieved_indices, relevant_indices, k)
        r_at_k = recall_at_k(retrieved_indices, relevant_indices, k)

        print(f"Precision@{k}: {p_at_k:.4f}")
        print(f"Recall@{k}: {r_at_k:.4f}")

        results.append({
            "query": query,
            "k": k,
            "precision": p_at_k,
            "recall": r_at_k
        })

In [ ]:
# Visualize the evaluation results
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Plot Precision@K and Recall@K for each query
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.lineplot(data=results_df, x="k", y="precision", hue="query", marker="o")
plt.title("Precision@K")
plt.xlabel("K")
plt.ylabel("Precision")

plt.subplot(1, 2, 2)
sns.lineplot(data=results_df, x="k", y="recall", hue="query", marker="o")
plt.title("Recall@K")
plt.xlabel("K")
plt.ylabel("Recall")

plt.tight_layout()
plt.show()

### Manual Evaluation Framework

In [ ]:
# Create a framework for manual evaluation
def manual_evaluation_template(query, retrieved_results):
    """
    Generate a template for manual evaluation of retrieved results.

    Args:
        query (str): The search query
        retrieved_results (list): List of retrieved results

    Returns:
        str: Evaluation template
    """
    template = f"""# Manual Evaluation for Query: '{query}'

## Evaluation Criteria:
- Relevance: How relevant is the result to the query? (1-5)
- Faithfulness: Does the result contain accurate information? (1-5)
- Completeness: Does the result provide complete information? (1-5)

## Results:
"""

    for i, result in enumerate(retrieved_results):
        template += f"""
### Result {i+1}:
Summary: {result['summary']}
Text: {result['text'][:200]}...

- Relevance: [Score 1-5]
- Faithfulness: [Score 1-5]
- Completeness: [Score 1-5]
- Comments: [Add any comments here]

"""

    return template